In [ ]:
# Atualiza dados
from app.services.pipeline import coletar_dados
from app.settings import ANALISES
import pickle
import sys
import os
from time import perf_counter

# BASE_DIR = os.path.dirname(os.path.abspath(__file__))
# if BASE_DIR not in sys.path:
#     sys.path.insert(0, BASE_DIR)


username = "felipe.cruz"
password = "#Gladoscruz.9851"

for analise in ANALISES:
	dfs = coletar_dados(username, password, analise)
	with open(f"snapshot_{analise}.pkl", "wb") as f:
		pickle.dump(dfs, f)
	print(f"Snapshot salvo para {analise}.")

['Item', 'Familia', '']
Extractor criado com sucesso!
ordens_LE FABRI02 concluída em 11.91s (1 / 119)
conf_VSN concluída em 6.30s (2 / 119)
consPES ACABA concluída em 20.80s (3 / 119)
ordens_ESTOF UL concluída em 22.74s (4 / 119)
conf_VRG concluída em 18.36s (5 / 119)
consPORTAS concluída em 13.05s (6 / 119)
consCC FABRI01 concluída em 25.70s (7 / 119)
conf_IDA concluída em 8.26s (8 / 119)
apoio_compras_VSN concluída em 28.45s (9 / 119)
apoio_compras_FNC concluída em 7.75s (10 / 119)
apoio_compras_VRG concluída em 3.31s (11 / 119)
consMETALURGIA concluída em 35.28s (12 / 119)
ordens_CAPAS LE concluída em 37.47s (13 / 119)
apoio_compras_FNC concluída em 15.85s (14 / 119)
ordens_PLANEJADOS concluída em 15.95s (15 / 119)
consCADEIRAS concluída em 12.77s (16 / 119)
consCAPAS LE concluída em 43.53s (17 / 119)
ordens_PORTAS concluída em 11.16s (18 / 119)
conf_FNC concluída em 32.78s (19 / 119)
ordens_UL ACABA01 concluída em 19.57s (20 / 119)
ordens_ACESSORIOS concluída em 10.83s (21 / 119)
c

In [ ]:
#Estoque Rejeitado
import pandas as pd
print (dfs.keys())
est_r = dfs.get("estoque_R").copy()
print(est_r.columns.to_list())
est_r = est_r[["Item", "Qtde."]]

dict_keys(['estoque_R', 'conf', 'cons', 'ordens', 'apoio_compras', 'estoque'])
['Item', 'Descrição', 'FAM', 'Sit.', 'Local - Sit', 'Qtde.', 'Unidade', 'Qt.Reser', 'PE', 'Endereço', 'Local Prod.', 'Sofre Baixa?', 'Lote Múltiplo', 'Local Padrão', 'Família']


In [2]:
#Carerga Snapshot

import pickle

analise = "compra por necessidade"
# analise = "Compra est NEC conf"
dfs = {}
with open(f"snapshot_{analise}.pkl", "rb") as f:
	dfs = pickle.load(f)
print(f"Snapshot carregado para {analise}.")
print(dfs.keys())

Snapshot carregado para compra por necessidade.
dict_keys(['conf', 'apoio_compras', 'ordens', 'cons', 'estoque', 'estoque_R'])


In [3]:
# Cálculo com grafos
import pandas as pd


print(f"{dfs.keys()}\n")


def sanitizar_dataframe(df, limite=0.8):
    df = df.copy()

    for col in df.columns:
        serie = df[col].astype(str).str.strip()

        tentativa_data = pd.to_datetime(
            serie, errors="coerce", dayfirst=True, format="%d/%m/%Y"
        )
        if tentativa_data.notna().mean() > limite:
            df[col] = tentativa_data
            continue

        serie_num = serie.str.replace(".", "", regex=False).str.replace(
            ",", ".", regex=False
        )
        tentativa_num = pd.to_numeric(serie_num, errors="coerce")
        if tentativa_num.notna().mean() > limite:
            df[col] = tentativa_num
            continue

        df[col] = serie.replace({"": None})

    return df


def calc_data(dfs):

    ## Ajuste Ordens ##
    ordens = sanitizar_dataframe(dfs.get("ordens"))
    ordens = ordens[
        [
            "Cliente",
            "Fábrica",
            "Ordem",
            "Pedido",
            "Item",
            "Saldo",
            "Representante",
            "Entrega Pedido",
            "Data Abertura",
        ]
    ]
    ordens = ordens.rename(columns={"Ordem": "Ordem Prod", "Saldo": "Saldo Prod"})

    colunas = ["Entrega Pedido", "Data Abertura"]
    for col in colunas:
        ordens[col] = (
            ordens[col]
            .astype(str)
            .str.strip()
            .str.replace(r"[^\d]", "", regex=True)
            .pipe(lambda s: pd.to_datetime(s, format="%d%m%Y", errors="coerce"))
        )

    ## Ajuste Consumo ##
    consumo = sanitizar_dataframe(dfs.get("cons"))
    consumo["Item"] = consumo["Item"].str.split("-").str[0].str.strip()
    consumo = consumo[
        [
            "Tipo",
            "Item",
            "Baixa",
            "Consumo",
            "Local Prod.",
            "OP",
            "Familia",
            "Den. Item",
        ]
    ]
    consumo = consumo.rename(columns={"OP": "Ordem Cons"})

    ######## Item Pai ##########
    consumo["item_pai"] = consumo["Ordem Cons"].map(
        ordens.set_index("Ordem Prod")["Item"]
    )

    ######## Merge 1: traz Pedido/Cliente/etc via Ordem Cons ##########
    consumo = consumo.merge(
        ordens[
            [
                "Cliente",
                "Fábrica",
                "Ordem Prod",
                "Pedido",
                "Saldo Prod",
                "Representante",
                "Entrega Pedido",
                "Data Abertura",
            ]
        ].rename(columns={"Ordem Prod": "Ordem", "Saldo Prod": "Saldo"}),
        left_on="Ordem Cons",
        right_on="Ordem",
        how="left",
    )

    ######## Merge 2: Ordem Prod do componente ##########
    ordens_op = ordens[["Item", "Ordem Prod", "Pedido"]].copy()

    consumo_com_pedido = consumo[consumo["Pedido"] > 0]
    consumo_sem_pedido = consumo[consumo["Pedido"] == 0]

    consumo_com_pedido = consumo_com_pedido.merge(
        ordens_op[ordens_op["Pedido"] > 0],
        on=["Item", "Pedido"],
        how="left",
    )

    consumo_sem_pedido = consumo_sem_pedido.merge(
        ordens_op.drop_duplicates("Item")[["Item", "Ordem Prod"]],
        on="Item",
        how="left",
    )

    consumo = pd.concat([consumo_com_pedido, consumo_sem_pedido]).sort_index()

    ######## Ordena as colunas ##########
    consumo = consumo[
        [
            "Tipo",
            "Ordem Prod",
            "Item",
            "Consumo",
            "Ordem Cons",
            "item_pai",
            "Saldo",
            "Pedido",
            "Representante",
            "Entrega Pedido",
            "Local Prod.",
            "Familia",
            "Cliente",
            "Fábrica",
            "Ordem",
            "Data Abertura",
            "Baixa",
            "Den. Item",
        ]
    ]

    ######## Calculos Baseados em estoque ##########
    estoque = sanitizar_dataframe(dfs.get("estoque"))
    consumo["estoque"] = (
        consumo["Item"].map(estoque.groupby("Item")["Qtde."].sum()).fillna(0)
    )

    from tqdm import tqdm

    ######## Propagação dos atributos da raiz ##########

    CAMPOS_RAIZ = [
        "Cliente",
        "Fábrica",
        "Pedido",
        "Representante",
        "Entrega Pedido",
        "Data Abertura",
        "Saldo",
        "Baixa",
    ]
    CAMPOS_RAIZ_COMPLETO = ["Item Final"] + CAMPOS_RAIZ

    itens_existentes = set(consumo["Item"].unique())
    raizes = set(consumo["item_pai"].dropna()) - itens_existentes

    # Nível 0: item_pai aqui É o produto final (raiz real)
    cache_raiz = {}
    for _, row in (
        consumo[consumo["item_pai"].isin(raizes)]
        .drop_duplicates("Ordem Cons")
        .iterrows()
    ):
        cache_raiz[row["Ordem Cons"]] = {
            "Item Final": row["item_pai"],
            **row[CAMPOS_RAIZ].to_dict(),
        }

    # Mapa Ordem Prod → [(Ordem Cons, Pedido_Ordem, Pedido_Contexto)]
    mapa_raw = (
        ordens[["Ordem Prod", "Item", "Pedido"]]
        .rename(columns={"Pedido": "Pedido_Ordem"})
        .merge(
            consumo[["Item", "Ordem Cons", "Pedido"]].drop_duplicates(
                ["Item", "Ordem Cons"]
            ),
            on="Item",
        )
    )

    mapa_op_para_pais = {}
    for _, row in mapa_raw.iterrows():
        op = row["Ordem Prod"]
        if op not in mapa_op_para_pais:
            mapa_op_para_pais[op] = []
        mapa_op_para_pais[op].append(
            (row["Ordem Cons"], row["Pedido_Ordem"], row["Pedido"])
        )

    def achar_pai_no_cache(ordem_cons_filho, pedido_filho):
        pais = mapa_op_para_pais.get(ordem_cons_filho, [])
        pais_no_cache = [(oc, po, pc) for oc, po, pc in pais if oc in cache_raiz]
        if not pais_no_cache:
            return None
        if len(pais_no_cache) == 1:
            return cache_raiz[pais_no_cache[0][0]]
        match = [(oc, po, pc) for oc, po, pc in pais_no_cache if po == pc]
        if len(match) == 1:
            return cache_raiz[match[0][0]]
        match2 = [(oc, po, pc) for oc, po, pc in pais_no_cache if pc == pedido_filho]
        if match2:
            return cache_raiz[match2[0][0]]
        return cache_raiz[pais_no_cache[0][0]]

    # BFS top-down
    for nivel in range(1, 12):
        pendentes = consumo[~consumo["Ordem Cons"].isin(cache_raiz)].drop_duplicates(
            "Ordem Cons"
        )
        novos = 0
        for _, row in tqdm(
            pendentes.iterrows(),
            total=len(pendentes),
            desc=f"Nível {nivel}",
            unit="ordem",
        ):
            resultado = achar_pai_no_cache(row["Ordem Cons"], row["Pedido"])
            if resultado:
                cache_raiz[row["Ordem Cons"]] = resultado
                novos += 1
        print(f"Nível {nivel}: {novos} novas entradas resolvidas")
        if novos == 0:
            break
        
    raiz_df = (
        consumo["Ordem Cons"]
        .map(cache_raiz)
        .apply(
            lambda x: (
                x if isinstance(x, dict) else {c: None for c in CAMPOS_RAIZ_COMPLETO}
            )
        )
    )
    raiz_df = pd.DataFrame(raiz_df.tolist(), index=consumo.index)
    raiz_df.columns = [f"raiz_{c}" for c in CAMPOS_RAIZ_COMPLETO]
    consumo = pd.concat([consumo, raiz_df], axis=1)

    #### Gera Excel ####
    print("gerar excel")
    csv_path = "CSV/"
    consumo.to_excel(csv_path + "consumo.xlsx", index=False)

    
    print("Concluido")


calc_data(dfs)

dict_keys(['conf', 'apoio_compras', 'ordens', 'cons', 'estoque', 'estoque_R'])



Nível 1: 100%|██████████| 6933/6933 [00:00<00:00, 51337.61ordem/s]


Nível 1: 5998 novas entradas resolvidas


Nível 2: 100%|██████████| 935/935 [00:00<00:00, 49138.86ordem/s]


Nível 2: 927 novas entradas resolvidas


Nível 3: 100%|██████████| 8/8 [00:00<00:00, 8019.70ordem/s]


Nível 3: 8 novas entradas resolvidas


Nível 4: 0ordem [00:00, ?ordem/s]


Nível 4: 0 novas entradas resolvidas
gerar excel
Concluido


In [5]:
# Gerar apoio
import pickle
from pathlib import Path
import pandas as pd

analise = "Compra est NEC conf"
dfs = {}
with open(f"snapshot_{analise}.pkl", "rb") as f:
    dfs = pickle.load(f)
print(f"Snapshot carregado para {analise}.")
print(dfs.keys())


from os import sep

apoio_comp = dfs.get("apoio_compras").copy()

# =========================
# Busca histórico de consumo quarto mes completo
# =========================
from app.services.processor import sanitizar_dataframe
from datetime import datetime
from dateutil.relativedelta import relativedelta

data = datetime.now() - relativedelta(months=4)
mes = data.month
ano = data.year
arquivo_apont = Path(f"apont-{ano}-{mes:02d}.csv")
df_apont = pd.read_csv(arquivo_apont, sep=";", decimal=",", encoding="utf-8-sig")

df_apont = sanitizar_dataframe(df_apont)

df_apont = df_apont[["Item", "Qtde."]].groupby("Item", as_index=False).sum()

nome_col = f"{ano}-{mes:02d}"

apoio_comp[nome_col] = (
    apoio_comp["Item"].map(df_apont.set_index("Item")["Qtde."]).fillna(0)
)

if nome_col in apoio_comp.columns:
    col = apoio_comp.pop(nome_col)
    apoio_comp.insert(2, nome_col, col)
else:
    apoio_comp.insert(2, nome_col, pd.NA)
apoio_comp = apoio_comp.reindex(columns=apoio_comp.columns.drop("Baixa").tolist() + ["Baixa"])
apoio_comp.to_csv(
    "CSV/apoio.csv", index=False, sep=";", decimal=",", encoding="utf-8-sig"
)

conf = dfs.get("conf").copy()
conf.to_csv("CSV/conf.csv", index=False, sep=";", encoding="utf-8-sig")

Snapshot carregado para Compra est NEC conf.
dict_keys(['conf', 'apoio_compras'])


In [6]:
# Tratar dados de entrega

import pandas as pd

# Caminho do arquivo
caminho = r"C:\Users\engli\OneDrive\Área de Trabalho\Analise_compras\CSV\conf.csv"


def tratar_csv():
    # ============================================================
    # LEITURA
    # ============================================================
    df = pd.read_csv(caminho, sep=None, engine="python")

    # Referência das colunas
    col_a = df.columns[0]  # Coluna A -> Item
    col_d = df.columns[3]  # Coluna D
    col_f = df.columns[5]  # Coluna F -> Data
    col_l = df.columns[11]  # Coluna L 

    # ============================================================
    # 1) MANTÉM APENAS D == 11
    # ============================================================
    df = df[df[col_d].astype(str).str.strip() == "11"]
    df = df[df[col_l].notna() & (df[col_l].astype(str).str.strip() != "")]

    # ============================================================
    # 2) CONVERTE A COLUNA F PARA DATA
    # ============================================================
    df[col_f] = pd.to_datetime(df[col_f], errors="coerce", dayfirst=True)

    # ============================================================
    # 3) REMOVE LINHAS SEM DATA VÁLIDA
    # ============================================================
    df = df[df[col_f].notna()]

    # Segurança extra:
    # remove valores que viraram NaT após conversão
    df = df.dropna(subset=[col_f])

    # ============================================================
    # 4) ORDENA PELOS MAIS RECENTES
    # ============================================================
    df = df.sort_values(by=col_f, ascending=False)

    # ============================================================
    # 5) MANTÉM SOMENTE OS 5 MAIS RECENTES POR ITEM
    # ============================================================
    df = df.groupby(col_a, group_keys=False).head(5)

    # ============================================================
    # 6) ORDENAÇÃO FINAL
    # ============================================================
    df = df.sort_values(by=[col_a, col_f], ascending=[True, False])

    # ============================================================
    # 7) SALVA O RESULTADO
    # ============================================================
    df.to_csv(caminho, index=False, sep=";")

    print("Arquivo tratado com sucesso.")


tratar_csv()

Arquivo tratado com sucesso.
